In [1]:
import pandas as pd
from events_data_core.stock_data_provider import get_stock_data

ticker = 'LKOH'
stock_data = get_stock_data(ticker)
stock_data = stock_data[stock_data['DATE'] > '2022-01-01']

stock_data.head()

,DATE,OPEN,HIGH,LOW,CLOSE,VOL
5001,2022-01-03,6592.0,6705.5,6586.0,6683.0,610921.0
5002,2022-01-04,6688.0,6741.0,6627.0,6728.0,551239.0
5003,2022-01-05,6715.5,6755.0,6501.0,6522.0,972892.0
5004,2022-01-06,6530.0,6706.0,6459.0,6699.0,906403.0
5005,2022-01-10,6730.0,6880.0,6650.5,6775.0,1238121.0


In [2]:
import duckdb

# подключаемся (in-memory, без файла)
con = duckdb.connect()

# читаем два csv
con.execute("""
            CREATE TABLE event_tags AS
            SELECT *
            FROM read_csv_auto('../data/db/event_tags.csv');
            """)

con.execute("""
            CREATE TABLE events AS
            SELECT *
            FROM read_csv_auto('../data/db/events.csv');
            """)

# извлекаем только санкционные события
query = """
        SELECT e.*
        FROM events e
                 JOIN event_tags t ON e.id = t.event_id
        WHERE t.tag_code = 'SANCTIONS'
          and e.date_start > '2022-01-01'
        """

sanctions_df = con.execute(query).df()
sanctions_df

,id,date_start,date_end,event
0,017f1eba-7c00-4bed-8a0c-e5ee2c76f009,2022-02-21,2022-02-23,Принятие первого пакета санкций против России.
1,017f2b9a-7200-4fca-a35a-c68140153c96,2022-02-24,2022-02-25,Принятие второго пакета санкций против России.
2,017f5c86-fc00-47eb-8988-aeb8b647cc14,2022-02-26,2022-03-14,Принятие третьего пакета санкций против России.
3,017fbe5f-7000-4aa7-b4d4-3688dbe73a96,2022-03-15,2022-04-04,Принятие четвертого пакета санкций против России.
4,01808c5d-f000-4f60-85eb-0aa34a373e1a,2022-04-05,2022-06-02,Принятие пятого пакета санкций против России.
5,01819302-7400-4bdb-8219-df4aec4b9fc7,2022-06-03,2022-07-15,Принятие шестого пакета санкций против России.
6,0182df2c-7200-4a21-96e5-597baa799d81,2022-07-21,2022-10-04,Принятие седьмого пакета санкций против России.
7,01845ed6-7800-41fd-af56-a56104072a56,2022-10-06,2022-12-15,Принятие 8 пакета санкций против России.
8,0185cc79-fc00-46ca-8a45-fd3c3a0342f8,2022-12-16,2023-02-24,Принят 9 пакет санкций против России.
9,0187b08f-7400-42c0-86b2-cb3b9c32c53a,2023-02-25,2023-06-21,Принятие 10 пакета санкций против России.


In [3]:
from events_data_core.plot_utils import plot_2d_events

app = plot_2d_events(stock_data['DATE'], stock_data['CLOSE'], sanctions_df)

app.run_server(debug=True)

In [4]:
# todo построить график относительно уровня инфляции, чтобы исключить влияния девальвации рубля